# KLUE-RoBERTa 한국어 70감정 분류 파인튜닝 v2

**이전 KoBERT 버전의 NumPy 충돌 문제 해결**  
- 모델: `klue/roberta-base` (KoBERT 대비 안정적, 성능 우수)
- 목표: accuracy 90%+

## 실행 순서
1. 런타임 → GPU 변경 (T4 이상)
2. 셀 순서대로 실행

In [ ]:
# ── 패키지 설치 (런타임 재시작 없이 동작하는 버전 조합) ──────────────────────
!pip install -q \
    'numpy==1.26.4' \
    'torch==2.3.0+cu121' --index-url https://download.pytorch.org/whl/cu121 \
    'transformers==4.44.2' \
    'accelerate==0.34.2' \
    'datasets==2.21.0' \
    'scikit-learn'
print('설치 완료')

In [ ]:
# ── Google Drive 마운트 ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = '/content/drive/MyDrive/감정분석모델/감정분석_70감정_추가증강.csv'
SAVE_PATH = '/content/drive/MyDrive/감정분석모델/klue-roberta-70emotion-v2'

In [ ]:
import numpy as np
import pandas as pd
import torch
import os
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback
)

os.environ['WANDB_DISABLED'] = 'true'
print(f'PyTorch: {torch.__version__}')
print(f'NumPy: {np.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음"}')

In [ ]:
# ── 데이터 로딩 및 전처리 ────────────────────────────────────────────────────
TEXT_COL = '문장 예시'
LABEL_COL = '감정명'

df = pd.read_csv(DATA_PATH, encoding='utf-8-sig').dropna(subset=[TEXT_COL, LABEL_COL])
print(f'원본: {len(df)}행, {df[LABEL_COL].nunique()}클래스')

# 충돌 텍스트: 다수결 레이블로 처리 (버리지 않음)
df['_norm'] = df[TEXT_COL].str.strip().str.lower()
conflict_mask = df.groupby('_norm')[LABEL_COL].transform('nunique') > 1
if conflict_mask.any():
    majority = (
        df[conflict_mask]
        .groupby('_norm')[LABEL_COL]
        .agg(lambda s: s.value_counts().index[0])
    )
    df.loc[conflict_mask, LABEL_COL] = df.loc[conflict_mask, '_norm'].map(majority)
    print(f'충돌 텍스트 {conflict_mask.sum()}행 → 다수결 처리')

df = df.drop_duplicates(['_norm', LABEL_COL]).reset_index(drop=True)
print(f'정제 후: {len(df)}행')

# 라벨 인코딩
encoder = LabelEncoder()
labels = encoder.fit_transform(df[LABEL_COL].tolist()).tolist()
texts = df[TEXT_COL].str.strip().tolist()

# 클래스별 분포 확인
dist = Counter(labels)
print(f'\n클래스당 평균: {np.mean(list(dist.values())):.0f}, '
      f'최소: {min(dist.values())}, 최대: {max(dist.values())}')

In [ ]:
# ── 데이터 분할 ───────────────────────────────────────────────────────────────
SEED = 42
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.15, random_state=SEED, stratify=labels
)
print(f'학습: {len(train_texts)}, 검증: {len(val_texts)}')

In [ ]:
# ── 토크나이저 & Dataset ──────────────────────────────────────────────────────
MODEL_NAME = 'klue/roberta-base'
MAX_LEN = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class EmotionDataset(Dataset):
    def __init__(self, texts, labels):
        self.enc = tokenizer(
            texts, truncation=True, padding='max_length',
            max_length=MAX_LEN, return_tensors='pt'
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.enc['input_ids'][idx],
            'attention_mask': self.enc['attention_mask'][idx],
            'labels': self.labels[idx],
        }

train_dataset = EmotionDataset(train_texts, train_labels)
val_dataset   = EmotionDataset(val_texts,   val_labels)
print('Dataset 생성 완료')

In [ ]:
# ── 모델 로딩 ─────────────────────────────────────────────────────────────────
id2label = {i: label for i, label in enumerate(encoder.classes_)}
label2id = {label: i for i, label in enumerate(encoder.classes_)}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(encoder.classes_),
    id2label=id2label,
    label2id=label2id,
)
print(f'파라미터 수: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# ── 평가 함수 ─────────────────────────────────────────────────────────────────
def compute_metrics(pred):
    labels = pred.label_ids
    preds  = np.argmax(pred.predictions, axis=1)
    return {
        'accuracy': float(accuracy_score(labels, preds)),
        'f1':       float(f1_score(labels, preds, average='weighted')),
    }

In [ ]:
# ── 학습 설정 ─────────────────────────────────────────────────────────────────
from transformers import DataCollatorWithPadding

training_args = TrainingArguments(
    output_dir='/content/roberta_checkpoints',
    num_train_epochs=10,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_steps=200,
    lr_scheduler_type='cosine',
    label_smoothing_factor=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    greater_is_better=True,
    save_total_limit=2,
    logging_steps=100,
    fp16=True,
    seed=42,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

In [ ]:
# ── 학습 시작 ─────────────────────────────────────────────────────────────────
trainer.train()

In [ ]:
# ── 최종 평가 ─────────────────────────────────────────────────────────────────
results = trainer.evaluate()
print(f"\n✅ 최종 정확도: {results['eval_accuracy']:.4f}")
print(f"✅ 최종 F1:     {results['eval_f1']:.4f}")

In [ ]:
# ── 모델 저장 (Google Drive) ───────────────────────────────────────────────────
import json
os.makedirs(SAVE_PATH, exist_ok=True)
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

# 레이블 클래스 저장
with open(f'{SAVE_PATH}/label_classes.json', 'w', encoding='utf-8') as f:
    json.dump(encoder.classes_.tolist(), f, ensure_ascii=False, indent=2)

# 성능 지표 저장
with open(f'{SAVE_PATH}/metrics.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f'저장 완료: {SAVE_PATH}')

In [ ]:
# ── 예측 예시 ─────────────────────────────────────────────────────────────────
from transformers import pipeline

classifier = pipeline(
    'text-classification',
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1,
    top_k=3,
)

test_sentences = [
    '오늘 너무 행복해서 눈물이 날 것 같아',
    '그 말 듣고 진짜 화가 많이 났어',
    '혼자 있으니까 외롭고 쓸쓸하다',
    '내일 발표가 있어서 너무 긴장돼',
]

for sent in test_sentences:
    preds = classifier(sent)[0]
    top = preds[0]
    print(f'입력: {sent}')
    print(f'  → {top["label"]} ({top["score"]:.3f})')
    print()

## 성능이 여전히 부족하다면?

### 옵션 A: 더 큰 모델
```python
MODEL_NAME = 'klue/roberta-large'  # 파라미터 3배, 속도 느림
```

### 옵션 B: KcELECTRA
```python
MODEL_NAME = 'snunlp/KcELECTRA-small-v2022'  # 가볍고 한국어 특화
```

### 옵션 C: 계층적 분류
- 1단계: 대분류 (긍정/부정/중립 or 감정군)
- 2단계: 각 군 내에서 세부 감정 분류
- 70클래스를 한 번에 분류하는 것보다 오류 감소

### 옵션 D: 앙상블
```python
# SVM + RoBERTa 확률 평균
final_pred = 0.3 * svm_prob + 0.7 * roberta_prob
```